# 02 · Baselines: persistence and ARX (system identification)

**Goal:** build the baselines that every fancy model must beat, and read an ARX model as a set of
learned **impulse responses**.

**Time:** ~45 min · **You'll write:** `persistence`, `arx_features`, `ridge`

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

from biotwin.data.dataset import Cohort, standard_sets, LH, H
from biotwin.models.baselines import ARX, Persistence
cohort = Cohort.load("data/cohort.npz")
S = standard_sets(cohort, seed=0)
tr, va, te = S["train"], S["val"], S["test_eval"]
rmse_h = lambda p, w: (p - w["y"]).pow(2).mean(0).sqrt()   # RMSE at each horizon

## Exercise 1 — persistence

"Glucose stays where it is": $\hat y(t_0 + k) = y(t_0 - 1)$ for every $k$. It's the baseline
that's embarrassingly hard to beat at short horizons.


<details><summary><b>Hint</b></summary>

`cgm_hist[:, -1:]` has shape `[N, 1]`; `.expand(-1, H)` repeats it without copying.

</details>

In [ ]:
def persistence(cgm_hist, H):
    # TODO: return [N, H]
    raise NotImplementedError

def check(f):
    close(f(te["cgm_hist"], H), Persistence().predict(te), what="persistence")

persistence = use(persistence, sol.persistence, check)

## 2. ARX

An ARX model predicts the change from the last reading as a **linear** function of recent CGM
and of the past and planned inputs. One separate linear model per horizon $k$ ("direct
multi-horizon"):

$$y(t_0+k) - y(t_0-1) = \sum_{j=1}^{n_a} a_{k,j}\,[y(t_0-1-j) - y(t_0-1)] + \sum_i b_{k,i}\, u_i + \sum_i c_{k,i}\, r_i + d_k$$

Features for one window (all concatenated):
1. `ar`: the `na` CGM samples *before* the last one, minus the last one, divided by 50;
2. `ins`: insulin minus basal, from `LH - n_in` to the end of the window (past + planned);
3. `cho`: carbs divided by 10, over the same range;
4. a column of ones.

The scalings just make the columns comparable in size.

## Exercise 2 — ARX features


<details><summary><b>Hint</b></summary>

`g[:, -na-1:-1] - g[:, -1:]` gives the AR part. `w['ins'][:, LH-n_in:] - w['basal'][:, None]` gives the insulin part. Return float64 (`.double()`): the normal equations are ill-conditioned in float32.

</details>

In [ ]:
def arx_features(w, na, n_in, LH):
    # TODO: return [N, na + 2*(n_in + H) + 1] float64
    raise NotImplementedError

def check(f):
    w = {k: v[:500] for k, v in tr.items()}
    close(f(w, 12, 48, LH), ARX(na=12, n_in=48).features(w), what="features")

arx_features = use(arx_features, sol.arx_features, check)

## Exercise 3 — ridge regression

Solve $\min_W \|XW - Y\|^2 + \lambda\|W\|^2$ for all 12 horizons at once:
$\hat W = (X^\top X + \lambda I)^{-1} X^\top Y$.


<details><summary><b>Hint</b></summary>

Use `torch.linalg.solve(A, B)` rather than forming an inverse: it's faster and more stable.

</details>

In [ ]:
def ridge(X, Y, lam):
    # TODO
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    X, Y = torch.randn(200, 10, generator=g, dtype=torch.float64), torch.randn(200, 3, generator=g, dtype=torch.float64)
    # independent check: ridge == least squares on the augmented system [X; sqrt(lam) I]
    Xa = torch.cat([X, 3.0 ** 0.5 * torch.eye(10, dtype=X.dtype)])
    Ya = torch.cat([Y, torch.zeros(10, 3, dtype=Y.dtype)])
    close(f(X, Y, 3.0), torch.linalg.lstsq(Xa, Ya).solution, tol=1e-8, what="W")

ridge = use(ridge, sol.ridge, check)

## 3. Fit, choose λ on validation patients, and test

In [ ]:
Xtr = arx_features(tr, 12, 48, LH)
Ytr = (tr["y"] - tr["cgm_hist"][:, -1:]).double()
Xva, Xte = arx_features(va, 12, 48, LH), arx_features(te, 12, 48, LH)
pred = lambda W, X, w: (X @ W).float() + w["cgm_hist"][:, -1:]

lams = [0.01, 0.1, 1, 10, 100, 1000, 1e4]
val_rmse = []
for lam in lams:
    W = ridge(Xtr, Ytr, lam)
    val_rmse.append(rmse_h(pred(W, Xva, va), va)[-1].item())
best = lams[int(np.argmin(val_rmse))]
plt.figure(figsize=(5, 3)); plt.semilogx(lams, val_rmse, "o-"); plt.xlabel("λ"); plt.ylabel("val RMSE 60 min"); plt.show()
print("best λ:", best)

W = ridge(Xtr, Ytr, best)
mins = np.arange(1, H + 1) * 5
plt.plot(mins, rmse_h(persistence(te["cgm_hist"], H), te), "o-", label="persistence")
plt.plot(mins, rmse_h(pred(W, Xte, te), te), "o-", label="ARX")
plt.xlabel("horizon (min)"); plt.ylabel("test RMSE"); plt.legend(); plt.show()
print(f"60 min: persistence {rmse_h(persistence(te['cgm_hist'], H), te)[-1]:.1f}, ARX {rmse_h(pred(W, Xte, te), te)[-1]:.1f}")

## 4. Reading the model: learned impulse responses

The coefficients $b_{k,i}$ say how much one unit of insulin at time $i$ moves glucose at the
target time. Plotted against "minutes before the target", **that's a learned impulse
response.** The same goes for carbs.

In [ ]:
na, n_in = 12, 48
k = H - 1                                   # the 60-min-ahead model
b_ins = W[na:na + n_in + H, k]              # insulin coefficients
b_cho = W[na + n_in + H:na + 2 * (n_in + H), k] / 10
lag_min = (LH + k - (LH - n_in + np.arange(n_in + H))) * 5   # minutes between input and target
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
a1.plot(lag_min, b_ins, "o-", ms=3); a1.set_title("insulin → glucose (mg/dL per U)")
a2.plot(lag_min, b_cho, "o-", ms=3, c="tab:orange"); a2.set_title("carbs → glucose (mg/dL per g)")
for a in (a1, a2):
    a.axhline(0, c="k", lw=0.8); a.set_xlabel("minutes before the target"); a.invert_xaxis()
plt.tight_layout(); plt.show()

Things to notice:
- The insulin response is **negative**, with the strongest effect about 1.5 h before the target:
  the delayed, low-pass effect from notebook 00, recovered by plain least squares.
- At 3–5 h lags the insulin coefficients turn slightly **positive**. Insulin doesn't raise glucose.
  This is **confounding**: boluses come with meals, so "insulin 4 hours ago" partly means "a meal 4
  hours ago". A regression only sees correlations. Keep this in mind for notebook 04.
- Carbs act **faster** than insulin. That's why eating before the bolus causes a spike.
- The curves are noisy and not smooth, because ridge only shrinks coefficients; it doesn't make
  them smooth. A physics model builds that shape in from the start, which is what notebook 03
  does.
- One set of coefficients serves **every** patient and every glucose level. ARX can't represent
  "insulin works better when glucose is high" (a bilinear effect) or "this patient is more
  sensitive".

## Quiz

Answer in your own words before opening the answers.

**1. Why predict y(t0+k) − y(t0−1) instead of y(t0+k) directly?**

<details><summary>Answer</summary>

Most of y(t0+k) is just the current level. Predicting the change removes that large, easy part, makes the target roughly zero-mean, and lets the coefficients focus on dynamics. It also builds persistence in as the zero-coefficient model.

</details>

**2. Direct multi-horizon (12 separate models) vs. one model applied recursively: what are the trade-offs?**

<details><summary>Answer</summary>

Direct models don't feed their own errors back into later steps, and each horizon gets its own fit. A recursive model has fewer parameters and its horizons are dynamically consistent, but errors compound. In closed form the direct version costs one solve with 12 right-hand sides.

</details>

**3. What happens to the validation curve as λ → 0 and as λ → ∞, and why?**

<details><summary>Answer</summary>

λ → 0: plain least squares with ~133 features, slightly higher variance (overfitting to the training patients). λ → ∞: all coefficients shrink to 0, which turns the model into persistence (high bias). The best value is in between.

</details>

**4. The learned insulin response is negative and slow. How would you check that it's a causal effect and not a correlation?**

<details><summary>Answer</summary>

Look for variation that breaks the meal–bolus link (missed boluses, timing jitter, correction boluses) and check that the effect holds there. Better, run an intervention in the simulator (same state, with and without a bolus). This is the question notebook 07 asks of the hybrid.

</details>

**5. Name two things ARX structurally cannot capture in this problem.**

<details><summary>Answer</summary>

1) The bilinear insulin×glucose interaction (insulin removes more glucose when glucose is high). 2) Differences between patients (one set of coefficients for all of them). Also saturations: renal clearance above a threshold, and the 39–600 sensor clip.

</details>